Idea: Use svg.py to create images that JUST contain the annotations (text and graphic), so each will be very small.
Then, we can use gimp or other tools to layer the two relevent thumb..bmp images and the annotation images. 
Then we can use gimp to shift from image to image, and calculate the difference and subtraction images.  

In [1]:
import numpy as np
import math
import matplotlib.pyplot as plt

In [2]:
#we use wand to probe images and (I think) draw the svg on a sky.
import wand
from wand.image import Image
from wand.drawing import Drawing

In [3]:
import svg

**File and directory paths and names**

In [4]:
#Numerical
intfilename="DroneShort1HalfDecimated.int.1"
outfilename="DroneShort1HalfDecimated.out.1"

#Image
IPath="/media/seth/CTAP/bitmaps-jobDS1HalfDecimatedFRAME"

In [5]:
def to6( n ):
    return "{!s:>06}".format(n)
def tothumb(n):
    return IPath+"/thumb"+to6(n)+".bmp"
def todatasvg(n):
    return IPath+"/data"+to6(n)+".svg"
tothumb(1234)

'/media/seth/CTAP/bitmaps-jobDS1HalfDecimatedFRAME/thumb001234.bmp'

**Numerical Data**

*Phase1a .int file output*

In [6]:
dtint = np.dtype([('frame', np.int32), 
               ('Rd','<i8'),('Rx',np.int16),('Ry',np.int16),
               ('Gd','<i8'),('Gx',np.int16),('Gy',np.int16),
               ('Bd','<i8'),('Bx',np.int16),('By',np.int16),
               ('rd','<i8'),('rx',np.int16),('ry',np.int16),
               ('gd','<i8'),('gx',np.int16),('gy',np.int16),
               ('bd','<i8'),('bx',np.int16),('by',np.int16),
               ('Rn',np.int32),('Gn',np.int32),('Bn',np.int32),
               ('rn',np.int32),('gn',np.int32),('bn',np.int32),
               ('stn',np.int32)]
             )

In [7]:
dtout = np.dtype([('cluster', np.int32), ('frame', np.int32),
                ('diff','<i8'),
                ('x',np.int16),('y',np.int16),
                ('score',np.float16)])

In [8]:
#np.loadtxt?

In [9]:
intdata = np.loadtxt(intfilename,converters=float,dtype=dtint)
nframes=len(intdata)
outdata = np.loadtxt(outfilename,converters=float,dtype=dtout)

In [10]:
#needed for svg dims
width=0
height=0
def setwh(n):
    img=Image(filename=tothumb(1))
    global width
    width=img.width
    global height
    height=img.height
setwh(1)
# this is used to scale graphics and text dims to our images' resolution
widthTo1920=math.ceil(float(width)/1920.0)

In [11]:
def getintrow(fn):
    row=np.array(intdata[fn-1])
    #print("row=",row)
    #print()
    t=intdata[fn-1].item(0)
    intdatarowstr = str(t[0])
    for z in range(1,24,3):
        intdatarowstr+="  "+str(t[z:z+3])
    intdatarowstr+="   "+str(t[24])
    return {"string": intdatarowstr, "row": row}

In [12]:
def getanyoutrow(frameno):
    # if there is a row in .out data with given frame number,
    # return list [  string of text to draw on image,
    #                the row (ndarray structure) for drawing circle on image ]
    # otherwise, None is returned
    row=outdata[  outdata[:]['frame']==frameno]
    #print("row=",row)
    if len(row) > 0:
        if len(row) > 1 :
            print("Somethings wrong. ", len(row), " have frame number ", frameno)
            print(row)
        r = row[0]
        #print(r)
        #print(type(r))
        ret={'string': str(r['cluster'])  + "     " 
             + str(r['frame']) + "      ("
             + str(r['x'] ) + ", " 
             + str(r['y']) + ")     " 
             + str(r['score']),
              'row': row[0]}
    else:
            ret=None
    #print(ret)
    #if ret != None :
    #    print(ret[0])
    #    print(type(ret[0]))
    return ret

**Geometry of Phase1a difference arrow visualization.**

In [13]:
def M(TH) :
    #The usual rotation by TH degrees matrix acting from the right.  In y-downward graphics coordinates, it rotates CLOCKWISE.
    return( np.array( [ [math.cos(math.pi*TH/180.), math.sin(math.pi*TH/180.)], [-math.sin(math.pi*TH/180.), math.cos(math.pi*TH/180.)] ] ) )

#Geometry of normalized unit arrows to show RGB changes
TH=30.0                  #angle of arrows away from vertical, and hands away from body
lcircr=0.1               #little circle radius
hslen=0.1                #length of each arrowhead back spike
# Unit Vectors
g1=np.array([0.0,1.0])   #unit lower case, down, green

#tiny vectors
tc=lcircr*np.array([0.0,1.0]) #radius (down, y dir of tiny circle, and foot of down unit arrow
def T(s) :
    return (tc + g1*s) #tail on tiny circle, head down by unit * s (scale, 0<=s<=1)

#Arrow body is (tc->T(s)*M..
def arrB(s) :
    return np.array([tc, T(s)])

TL=hslen*g1@M(180.0+TH) #coord of left hand rel to head
TR=hslen*g1@M(180.0-TH) #coord of right hand rel to head

#down dir Left arm LA(s) is (T(s)->TL(s))
def arrL(s) :
    return np.array([T(s), T(s)+TL])


#down dir Right arm LA(s) is (T(s)->TL(s))
def arrR(s) :
    return np.array([T(s), T(s)+TR])

def unit_up_arrow(s) :
    return -np.concat([arrB(s),arrL(s),arrR(s)])


In [14]:
#define scale of color value differences to pixel coords.
colvaldiv = float(128)
arrlen = 100*widthTo1920

In [15]:
#This defines the 6 arrow dirs (degrees clockwise from vertical) and colors, give a row from the .int file
def dispdata(row):
    #print(fn, row['frame'])
    return ( [ row['Rd'],   -30.0, [row['Rx'],row['Ry']], row['Rn' ], "red" ],
             [ row['Gd'],     0.0, [row['Gx'],row['Gy']], row['Gn' ], "green" ],
             [ row['Bd'],    30.0, [row['Bx'],row['By']], row['Bn' ], "blue" ],
             [ -row['rd'], -150.0, [row['rx'],row['ry']], row['rn' ], "red" ],
             [ -row['gd'],  180.0, [row['gx'],row['gy']], row['gn' ], "green" ],
             [ -row['bd'],  150.0, [row['rx'],row['by']], row['bn' ], "blue" ] ) 

In [16]:
#svg.Line?

**Now come some functions to append svg.py elements into a list**

In [17]:
#refactored from drdata()
def addarrowelts( svgelts, row ):
    data = dispdata(row)
    for r in data: 
        line = np.round( (arrlen*unit_up_arrow(r[0]/colvaldiv))@M(r[1]) + r[2]).astype(int)
        #print(line)
        for i in range(0,3):
            svgelts.append(svg.Line(stroke_opacity=1,
                x1=line[2*i][0], y1=line[2*i][1], 
                x2=line[2*i+1][0], y2=line[2*i+1][1],
                stroke_width=2*widthTo1920,
                style="stroke:"+r[4]))
        #for test/debugging
        #svgelts.append(svg.Circle(stroke_opacity=0.5,
        #    cx=r[2][0].astype(int), cy=r[2][1].astype(int), r="1px", fill_opacity=0.5,
        #    stroke_width="1px", style="stroke:"+r[4]+";opacity=0.5"))
        #r=data[0]
        #svgelts.append(svg.Rect(stroke_opacity=0.5,
        #    x=r[2][0].astype(int), y=r[2][1].astype(int), 
        #    width=0.5, height=0.5, fill_opacity=0.5,
        #    stroke_width="0.5", style="stroke:"+r[4]+";opacity=0.5"))
            svgelts.append(svg.Rect(#stroke_opacity=0.5,
                x=r[2][0].astype(int), y=r[2][1].astype(int), 
                width=0.5, height=0.5, fill_opacity=0.8,
                stroke_width="0", style="fill:"+r[4]+";opacity=0.8"))

In [18]:
llcapx=int(width/20)
llcapy=int(height-width/20)
llcapyabove=int(height-1.5*width/20)

In [19]:
def addtextelts( svgelts, frameno):
    text=getintrow(frameno)['string']
    svgelts.append(svg.Text(x=llcapx, y=llcapy, text=text,style="font-size:"+str(90*widthTo1920)+";fill:rgb( 150 150 90);stroke( 150 150 90)"))
    text=getanyoutrow(frameno)
    if(text):
        text=text['string']
        svgelts.append(svg.Text(x=llcapx, y=llcapyabove, text=text,style="font-size:"+str(90*widthTo1920)+";fill:orange"))

In [20]:
def add1bcircle(svgelts,data):
    svgelts.append(svg.Circle(cx=data['x'], cy=data['y'], r=str(30*widthTo1920),
                              fill_opacity=0,stroke_opacity=1,stroke_width=2*widthTo1920,style="stroke:cyan"))

***CRAZY When I used fill: colors like red, rgb(dd,dd,dd), RGB(dd dd dd), Imagemagick didn't overlay text on sky image, but I found #rrbbgg (hex) worked***

In [21]:
#debugging# makesvg_last_list=[]
def makesvg(frameno):
    svgelts=[]

    introw=getintrow(frameno)
    if introw :
        text=introw['string']
        svgelts.append(svg.Text(x=llcapx, y=llcapy, text=text, font_size=25*widthTo1920, word_spacing='exact',fill_opacity=1,stroke_opacity=1,
                                style="word_spacing=exact;font-size:"+str(25*widthTo1920)+";fill:#ffff00;stroke:#ffff00")) #;stroke:rgb( 150 150 90 )"))
        data=introw['row']
        addarrowelts(svgelts,data)

    outrow=getanyoutrow(frameno)
    if outrow :
        text=outrow['string']
        svgelts.append(svg.Text(x=llcapx, y=llcapyabove, text=text, font_size=25*widthTo1920, word_spacing='exact',fill_opacity=1,stroke_opacity=1,
                                style="word_spacing=exact;font-size:"+str(25*widthTo1920)+";fill:#00ffff;stroke:#00ffff")) # ;stroke:rgb( 0 255 255 )"))
        data=outrow['row']
        add1bcircle(svgelts,data)
    #debugging# for e in svgelts:
    #debugging#    print(e)
    #debuffing# global makesvg_last_list
    #debugging# makesvg_last_list = svgelts
    return svg.SVG(width=width,height=height,elements=svgelts)

In [27]:
image_format_sky_1ab = ".jpg"  #haven't paramterized quality of .jpg or other compression.
def datafilename(n):
    return "data"+to6(n)+".svg"
def imgwabfilename(n):
    return "imgwab"+to6(n)+image_format_sky_1ab
def makesvgfile(frameno):
    with open(datafilename(frameno), "w") as f:
        print(makesvg(frameno), file=f)
        print(f)

def alldatasvgs():
    for i in range(nframes):
        fn=i+1
        with open(datafilename(n), "w") as f:
            print(makesvg(fn), file=f)
        print("data ", fn, " saved.", end='\r')

def make_one_imgwab(fn):
    """Returns the wand (python Imagemagick API) image comprised
    of the thumb..fn..bmp movie frame and Phase 1a and b, if any,
    printed and graphically visualized on it.

    The directories where the thumb...jpg, .int (Phase1a data)
    and .out (Phase1b data) are configured in variables above,
    for now.

    :param fn: The frame number

    """
    makesvgfile(fn)
    dataimg=Image(background="transparent", filename=datafilename(fn))
    skyimg=Image(filename=tothumb(fn))
    skyimg.composite(dataimg)
    return skyimg
    
def make_save_one_imgwab(fn):
    skyimg=make_one_imgwab(fn)
    skyimg.save(filename=imgwabfilename(fn))

In [34]:
#foo=make_one_imgwab(206)
#foo